# Global table of test results

This notebook builds tables with the test results of model `Llama3.1-I`, split by recommendation method, combining:
- the `without_optimization` result, shown in the first row of each method;
- and all `with_optimization` results found for the configurations in `out/prompt_optimization/Llama3.1-I`.

The goal here is to provide a consolidated view by algorithm, covering all methods found in the current `out` directory.


In [1]:
import json
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display


In [2]:
def _is_project_root(candidate: Path) -> bool:
    return (candidate / "run_prompt_optimizer.py").exists() and (candidate / "out").exists()


def find_local_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if _is_project_root(candidate):
            return candidate

    descendant_hints = []
    for base in (start, *start.parents):
        descendant_hints.extend(
            [
                base / "prompt-optim-expl-rec" / "explainability-with-LLMs",
                base / "explainability-with-LLMs",
            ]
        )

    for candidate in descendant_hints:
        if _is_project_root(candidate):
            return candidate

    raise FileNotFoundError(
        "Could not locate the explainability-with-LLMs root. "
        "Run the notebook inside the project, from one of its subdirectories, or from the workspace root."
    )


def display_workspace_relative(path: Path, workspace_root: Path) -> str:
    resolved = path.resolve()
    try:
        return str(resolved.relative_to(workspace_root.resolve()))
    except ValueError:
        return str(resolved)


def display_path_string(path_str: str | None, workspace_root: Path) -> str | None:
    if not path_str:
        return path_str

    path = Path(path_str)
    if not path.is_absolute():
        return path_str

    return display_workspace_relative(path, workspace_root)

PROJECT_ROOT = find_local_project_root(Path.cwd().resolve())
WORKSPACE_ROOT = PROJECT_ROOT.parent
MODEL_NAME = "Llama3.1-I"
PROMPT_OPT_ROOT = PROJECT_ROOT / "out" / "prompt_optimization" / MODEL_NAME
TEST_ROOT = PROJECT_ROOT / "out" / "test_explainability"
WITHOUT_OPT_TEST_ROOT = TEST_ROOT / "without_optimization" / MODEL_NAME
WITH_OPT_TEST_ROOT = TEST_ROOT / "with_optimization" / MODEL_NAME

print(f"PROJECT_ROOT: {display_workspace_relative(PROJECT_ROOT, WORKSPACE_ROOT)}")
print(f"PROMPT_OPT_ROOT: {display_workspace_relative(PROMPT_OPT_ROOT, WORKSPACE_ROOT)}")
print(
    "WITHOUT_OPT_TEST_ROOT: "
    f"{display_workspace_relative(WITHOUT_OPT_TEST_ROOT, WORKSPACE_ROOT)}"
)
print(
    "WITH_OPT_TEST_ROOT: "
    f"{display_workspace_relative(WITH_OPT_TEST_ROOT, WORKSPACE_ROOT)}"
)


PROJECT_ROOT: explainability-with-LLMs
PROMPT_OPT_ROOT: explainability-with-LLMs/out/prompt_optimization/Llama3.1-I
WITHOUT_OPT_TEST_ROOT: explainability-with-LLMs/out/test_explainability/without_optimization/Llama3.1-I
WITH_OPT_TEST_ROOT: explainability-with-LLMs/out/test_explainability/with_optimization/Llama3.1-I


In [3]:
def lambda_to_float(lambda_name: str | None) -> float | None:
    if not lambda_name or not lambda_name.startswith("mmr_lambda_"):
        return None
    value = lambda_name.replace("mmr_lambda_", "")
    return float(value.replace("_", "."))


def find_named_parent(path: Path, prefix: str, default: str | None = None) -> str | None:
    for parent in path.parents:
        if parent.name.startswith(prefix):
            return parent.name
    return default


def discover_prompt_optimization_algorithms(prompt_opt_root: Path) -> list[str]:
    if not prompt_opt_root.exists():
        return []
    return sorted(path.name for path in prompt_opt_root.iterdir() if path.is_dir())


def load_test_metadata(metadata_path: Path) -> dict:
    return json.loads(metadata_path.read_text(encoding="utf-8"))


def discover_test_algorithms(test_root: Path) -> list[str]:
    if not test_root.exists():
        return []
    return sorted(path.name for path in test_root.iterdir() if path.is_dir())


def discover_without_optimization_results(test_root: Path, valid_algorithms: list[str]) -> pd.DataFrame:
    rows = []

    for algorithm in valid_algorithms:
        algorithm_dir = test_root / algorithm
        if not algorithm_dir.exists():
            continue

        for metadata_path in sorted(algorithm_dir.rglob("responses_metadata.json")):
            payload = load_test_metadata(metadata_path)
            args = payload.get("args", {})
            rows.append(
                {
                    "optimization_mode": "without_optimization",
                    "algorithm": algorithm,
                    "metric": payload.get("metric", args.get("metric", "metric")),
                    "metric_name": payload.get("metric_name", "METRIC"),
                    "metric_value": payload.get("metric_value"),
                    "repr_model": pd.NA,
                    "early_profile": pd.NA,
                    "mmr_lambda": pd.NA,
                    "lambda_value": pd.NA,
                    "mmr_pool": pd.NA,
                    "prompt_source": payload.get("prompt_source", "desconhecido"),
                    "llm_method": args.get("llm_method", "desconhecido"),
                    "n_users": payload.get("n_users"),
                    "time_to_explain": payload.get("time_to_explain"),
                    "best_prompt_path": display_path_string(payload.get("best_prompt_path"), WORKSPACE_ROOT),
                    "responses_metadata_path": display_workspace_relative(metadata_path, WORKSPACE_ROOT),
                }
            )

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)


def discover_with_optimization_results(test_root: Path, valid_algorithms: list[str]) -> pd.DataFrame:
    rows = []

    for algorithm in valid_algorithms:
        algorithm_dir = test_root / algorithm
        if not algorithm_dir.exists():
            continue

        for metadata_path in sorted(algorithm_dir.rglob("responses_metadata.json")):
            payload = load_test_metadata(metadata_path)
            args = payload.get("args", {})
            mmr_lambda = find_named_parent(metadata_path, "mmr_lambda_", None)

            rows.append(
                {
                    "optimization_mode": "with_optimization",
                    "algorithm": algorithm,
                    "metric": payload.get("metric", args.get("metric", "metric")),
                    "metric_name": payload.get("metric_name", "METRIC"),
                    "metric_value": payload.get("metric_value"),
                    "repr_model": find_named_parent(metadata_path, "repr_", pd.NA),
                    "early_profile": find_named_parent(metadata_path, "early_", pd.NA),
                    "mmr_lambda": mmr_lambda or pd.NA,
                    "lambda_value": lambda_to_float(mmr_lambda),
                    "mmr_pool": find_named_parent(metadata_path, "mmr_pool_", pd.NA),
                    "prompt_source": payload.get("prompt_source", "desconhecido"),
                    "llm_method": args.get("llm_method", payload.get("best_prompt_model", "desconhecido")),
                    "n_users": payload.get("n_users"),
                    "time_to_explain": payload.get("time_to_explain"),
                    "best_prompt_path": display_path_string(payload.get("best_prompt_path"), WORKSPACE_ROOT),
                    "responses_metadata_path": display_workspace_relative(metadata_path, WORKSPACE_ROOT),
                }
            )

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)


def build_consolidated_table(
    prompt_opt_root: Path,
    without_opt_test_root: Path,
    with_opt_test_root: Path,
) -> pd.DataFrame:
    preferred_columns = [
        "optimization_mode",
        "algorithm",
        "llm_method",
        "metric",
        "metric_name",
        "metric_value",
        "repr_model",
        "early_profile",
        "mmr_lambda",
        "lambda_value",
        "mmr_pool",
        "prompt_source",
        "n_users",
        "time_to_explain",
        "best_prompt_path",
        "responses_metadata_path",
    ]

    algorithms = sorted(
        set(discover_prompt_optimization_algorithms(prompt_opt_root))
        | set(discover_test_algorithms(without_opt_test_root))
        | set(discover_test_algorithms(with_opt_test_root))
    )
    without_opt = discover_without_optimization_results(without_opt_test_root, algorithms)
    with_opt = discover_with_optimization_results(with_opt_test_root, algorithms)

    rows = []
    for frame in (without_opt, with_opt):
        if frame.empty:
            continue
        rows.extend(frame.reindex(columns=preferred_columns).to_dict(orient="records"))

    if not rows:
        return pd.DataFrame(columns=preferred_columns)

    consolidated = pd.DataFrame(rows, columns=preferred_columns)
    consolidated["optimization_order"] = consolidated["optimization_mode"].map(
        {"without_optimization": 0, "with_optimization": 1}
    )
    consolidated = consolidated.sort_values(
        by=["algorithm", "metric", "optimization_order", "repr_model", "lambda_value", "mmr_pool"],
        na_position="last",
    ).reset_index(drop=True)

    return consolidated[preferred_columns]


In [4]:
consolidated_table = build_consolidated_table(
    prompt_opt_root=PROMPT_OPT_ROOT,
    without_opt_test_root=WITHOUT_OPT_TEST_ROOT,
    with_opt_test_root=WITH_OPT_TEST_ROOT,
)

if consolidated_table.empty:
    warnings.warn("No test result was found to build the global table.")
else:
    print(f"Rows in the consolidated table: {len(consolidated_table)}")
    for algorithm, algorithm_table in consolidated_table.groupby("algorithm", sort=False):
        print(f"\nAlgoritmo: {algorithm}")
        display(algorithm_table.reset_index(drop=True))


Rows in the consolidated table: 84

Algoritmo: bprmf


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,bprmf,Llama3.1-I,etd,ETD,0.792623,None,None,None,NaN,None,default,122,223.781048,None,explainability-with-LLMs/out/test_explainabili...
1,with_optimization,bprmf,Llama3.1-I,etd,ETD,0.814754,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,371.171364,out/prompt_optimization/Llama3.1-I/bprmf/etd/r...,explainability-with-LLMs/out/test_explainabili...
2,with_optimization,bprmf,Llama3.1-I,etd,ETD,0.818033,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,471.574602,out/prompt_optimization/Llama3.1-I/bprmf/etd/r...,explainability-with-LLMs/out/test_explainabili...
3,with_optimization,bprmf,Llama3.1-I,etd,ETD,0.812295,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,484.818186,out/prompt_optimization/Llama3.1-I/bprmf/etd/r...,explainability-with-LLMs/out/test_explainabili...
4,with_optimization,bprmf,Llama3.1-I,etd,ETD,0.834426,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,332.057089,out/prompt_optimization/Llama3.1-I/bprmf/etd/r...,explainability-with-LLMs/out/test_explainabili...
5,with_optimization,bprmf,Llama3.1-I,etd,ETD,0.848361,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,452.210245,out/prompt_optimization/Llama3.1-I/bprmf/etd/r...,explainability-with-LLMs/out/test_explainabili...
6,with_optimization,bprmf,Llama3.1-I,etd,ETD,0.804098,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,421.822154,out/prompt_optimization/Llama3.1-I/bprmf/etd/r...,explainability-with-LLMs/out/test_explainabili...
7,without_optimization,bprmf,Llama3.1-I,sep,SEP,0.644069,None,None,None,NaN,None,default,122,217.376059,None,explainability-with-LLMs/out/test_explainabili...
8,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.711800,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,212.449786,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,explainability-with-LLMs/out/test_explainabili...
9,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.711800,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,212.598656,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,explainability-with-LLMs/out/test_explainabili...



Algoritmo: item_knn


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,item_knn,Llama3.1-I,etd,ETD,0.778689,None,None,None,NaN,None,default,122,224.068505,None,explainability-with-LLMs/out/test_explainabili...
1,with_optimization,item_knn,Llama3.1-I,etd,ETD,0.840984,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,468.116610,out/prompt_optimization/Llama3.1-I/item_knn/et...,explainability-with-LLMs/out/test_explainabili...
2,with_optimization,item_knn,Llama3.1-I,etd,ETD,0.842047,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,492.441722,out/prompt_optimization/Llama3.1-I/item_knn/et...,explainability-with-LLMs/out/test_explainabili...
3,with_optimization,item_knn,Llama3.1-I,etd,ETD,0.818033,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,450.497600,out/prompt_optimization/Llama3.1-I/item_knn/et...,explainability-with-LLMs/out/test_explainabili...
4,with_optimization,item_knn,Llama3.1-I,etd,ETD,0.851639,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,486.702551,out/prompt_optimization/Llama3.1-I/item_knn/et...,explainability-with-LLMs/out/test_explainabili...
5,with_optimization,item_knn,Llama3.1-I,etd,ETD,0.812295,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,430.466688,out/prompt_optimization/Llama3.1-I/item_knn/et...,explainability-with-LLMs/out/test_explainabili...
6,with_optimization,item_knn,Llama3.1-I,etd,ETD,0.818033,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,450.141451,out/prompt_optimization/Llama3.1-I/item_knn/et...,explainability-with-LLMs/out/test_explainabili...
7,without_optimization,item_knn,Llama3.1-I,sep,SEP,0.589186,None,None,None,NaN,None,default,122,216.976493,None,explainability-with-LLMs/out/test_explainabili...
8,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.693893,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,207.053804,out/prompt_optimization/Llama3.1-I/item_knn/se...,explainability-with-LLMs/out/test_explainabili...
9,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.701002,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,216.124152,out/prompt_optimization/Llama3.1-I/item_knn/se...,explainability-with-LLMs/out/test_explainabili...



Algoritmo: ncf


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,ncf,Llama3.1-I,etd,ETD,0.767213,None,None,None,NaN,None,default,122,222.245019,None,explainability-with-LLMs/out/test_explainabili...
1,with_optimization,ncf,Llama3.1-I,etd,ETD,0.837705,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,406.145870,out/prompt_optimization/Llama3.1-I/ncf/etd/rep...,explainability-with-LLMs/out/test_explainabili...
2,with_optimization,ncf,Llama3.1-I,etd,ETD,0.837705,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,406.618049,out/prompt_optimization/Llama3.1-I/ncf/etd/rep...,explainability-with-LLMs/out/test_explainabili...
3,with_optimization,ncf,Llama3.1-I,etd,ETD,0.841803,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,444.620464,out/prompt_optimization/Llama3.1-I/ncf/etd/rep...,explainability-with-LLMs/out/test_explainabili...
4,with_optimization,ncf,Llama3.1-I,etd,ETD,0.837705,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,404.875024,out/prompt_optimization/Llama3.1-I/ncf/etd/rep...,explainability-with-LLMs/out/test_explainabili...
5,with_optimization,ncf,Llama3.1-I,etd,ETD,0.837705,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,406.816512,out/prompt_optimization/Llama3.1-I/ncf/etd/rep...,explainability-with-LLMs/out/test_explainabili...
6,with_optimization,ncf,Llama3.1-I,etd,ETD,0.837705,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,404.058072,out/prompt_optimization/Llama3.1-I/ncf/etd/rep...,explainability-with-LLMs/out/test_explainabili...
7,without_optimization,ncf,Llama3.1-I,sep,SEP,0.608446,None,None,None,NaN,None,default,122,216.910011,None,explainability-with-LLMs/out/test_explainabili...
8,with_optimization,ncf,Llama3.1-I,sep,SEP,0.711348,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,207.393923,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,explainability-with-LLMs/out/test_explainabili...
9,with_optimization,ncf,Llama3.1-I,sep,SEP,0.711348,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,207.789637,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,explainability-with-LLMs/out/test_explainabili...



Algoritmo: user_knn


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,user_knn,Llama3.1-I,etd,ETD,0.772131,None,None,None,NaN,None,default,122,225.370935,None,explainability-with-LLMs/out/test_explainabili...
1,with_optimization,user_knn,Llama3.1-I,etd,ETD,0.829508,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,464.307420,out/prompt_optimization/Llama3.1-I/user_knn/et...,explainability-with-LLMs/out/test_explainabili...
2,with_optimization,user_knn,Llama3.1-I,etd,ETD,0.808197,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,458.497143,out/prompt_optimization/Llama3.1-I/user_knn/et...,explainability-with-LLMs/out/test_explainabili...
3,with_optimization,user_knn,Llama3.1-I,etd,ETD,0.797541,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,379.580825,out/prompt_optimization/Llama3.1-I/user_knn/et...,explainability-with-LLMs/out/test_explainabili...
4,with_optimization,user_knn,Llama3.1-I,etd,ETD,0.822131,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,460.250184,out/prompt_optimization/Llama3.1-I/user_knn/et...,explainability-with-LLMs/out/test_explainabili...
5,with_optimization,user_knn,Llama3.1-I,etd,ETD,0.835246,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,459.914916,out/prompt_optimization/Llama3.1-I/user_knn/et...,explainability-with-LLMs/out/test_explainabili...
6,with_optimization,user_knn,Llama3.1-I,etd,ETD,0.827869,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,379.345260,out/prompt_optimization/Llama3.1-I/user_knn/et...,explainability-with-LLMs/out/test_explainabili...
7,without_optimization,user_knn,Llama3.1-I,sep,SEP,0.626153,None,None,None,NaN,None,default,122,216.094421,None,explainability-with-LLMs/out/test_explainabili...
8,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.685969,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,477.644687,out/prompt_optimization/Llama3.1-I/user_knn/se...,explainability-with-LLMs/out/test_explainabili...
9,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.615630,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,432.988393,out/prompt_optimization/Llama3.1-I/user_knn/se...,explainability-with-LLMs/out/test_explainabili...
